# SAM3 text-prompted tracking

Main research pipeline: prompt a video with a text query (`"dolphin"`) on frame 0, propagate masks across the whole video, and export a masklet overlay video.

Session order matters: `start_session` → `add_prompt` → `propagate_in_video` → `close_session` → `predictor.shutdown()`. `reset_session` is REQUIRED before switching to a different text prompt in the same session.

The checkpoint auto-downloads from the gated `facebook/sam3`/`facebook/sam3.1` repos: `load_env()` must find `HF_TOKEN`.

In [ ]:
import torch

from fish_segmentation.paths import find_repo_root, load_env
from fish_segmentation.sam3_utils import propagate_in_video
from fish_segmentation.video_io import load_all_frames_rgb

load_env()
ROOT = find_repo_root()

gpus_to_use = range(torch.cuda.device_count())
OUTPUTS = ROOT / "notebooks" / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


In [ ]:
from sam3.model_builder import build_sam3_video_predictor
from sam3.visualization_utils import (
    prepare_masks_for_visualization,
    save_masklet_video,
    visualize_formatted_frame_output,
)

predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12

video_path = ROOT / "data" / "test_media" / "dolphin_00.mp4"
video_frames_for_vis = load_all_frames_rgb(str(video_path))

In [ ]:
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=str(video_path),
    )
)
session_id = response["session_id"]

In [ ]:
prompt_text_str = "dolphin"
frame_idx = 0  # add a text prompt on frame 0

response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=frame_idx,
        text=prompt_text_str,
    )
)
out = response["outputs"]

In [ ]:
plt.close("all")
visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=["SAM 3 Dense Tracking outputs"],
    figsize=(6, 4),
)

In [ ]:
outputs_per_frame = propagate_in_video(predictor, session_id)

save_masklet_video(
    video_frames=video_frames_for_vis,
    outputs=outputs_per_frame,
    out_path=str(ROOT / "notebooks" / "outputs" / "output_tracking.mp4"),
    alpha=0.5,
    fps=15,
)

In [ ]:
from IPython.display import Video

Video(str(ROOT / "notebooks" / "outputs" / "output_tracking.mp4"), embed=True, width=640)

In [ ]:
# close the inference session to free its GPU resources
# (you may start a new session on another video)
_ = predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)

In [ ]:
# after all inference is done, shutdown the predictor
# to free up the multi-GPU process group
predictor.shutdown()